In [1]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

Current Project Root Directory set to: /Users/luca/Documents/GitHub/US_conflict_forecasting
Current Search Path set to: ['/Users/luca/Documents/GitHub/US_conflict_forecasting', '/Users/luca/Documents/GitHub/US_conflict_forecasting/1_Features/a_Spatial_Joins/2022', '/Users/luca/anaconda3/envs/ML_Lab/lib/python312.zip', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/lib-dynload', '', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages', '/Users/luca/Documents/GitHub/US_conflict_forecasting']


In [2]:
import pandas as pd
import folium
import geopandas as gpd
from shapely.geometry import Point

### Parsing ACLED Data 

In [3]:
#Load Data
ACLED_20_23_all = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/a_Input_Datasets/US2020_2023.csv'))


In [4]:
#Clean Data, filtering for PA
ACLED_22_PA = ACLED_20_23_all[(ACLED_20_23_all['admin1'] == 'Pennsylvania') & (ACLED_20_23_all['year'] == 2022)]

#Transform ACLED Data into GeoDataframe
#iterating over each row of the ACLED dataset to create list of point objects using lat and lon values
geometry = [Point(xy) for xy in zip(ACLED_22_PA['longitude'], ACLED_22_PA['latitude'])]

#creating new Geo dataframe using the Geodatapoints extracted in the previous step
gdf_ACLED_22_PA = gpd.GeoDataFrame(ACLED_22_PA, geometry=geometry)

### Parsing Vector Layer (ZIP Code Polygons)

In [5]:
#Parsing Geojson file 
gdf_ZIP_PA = gpd.read_file(os.path.join(project_root_dir, '2_Dataset/a_Input_Datasets/pa_pennsylvania_zip_codes_geo.min.json'))

### Overlaying Maps

In [6]:
#Defining center of map using bounds of the ZIP Code polygons --> calculating by averaging lat and long values to find midpoint
bounds = gdf_ZIP_PA.total_bounds
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

#Initialising map centrered around the calculated center 
map_PA = folium.Map(location=center, zoom_start=10)

#Adding ZIP Code polygon layer to initialised map 
folium.GeoJson(gdf_ZIP_PA, name="ZIP Codes PA").add_to(map_PA)

#Adding ACLED data points to initialised map 
for idx, row in ACLED_22_PA.iterrows():
    folium.Marker([row['latitude'], row['longitude']]).add_to(map_PA)


#View map (not loading map, takes forever)
#map_PA

In [ ]:
#View map (not loading map, takes forever)
map_PA

### Spatial Join 
**Extracting ZIP Code information for each conflict event**

In [26]:
spatial_join_PA = gpd.sjoin(gdf_ACLED_22_PA, gdf_ZIP_PA, how="inner", op='within')

/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3493: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):
/var/folders/4s/d5gptfss61l9fqz7rk3r5m880000gn/T/ipykernel_20968/743671876.py:1: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  spatial_join_PA = gpd.sjoin(gdf_ACLED_22_PA, gdf_ZIP_PA, how="inner", op='within')


In [29]:
#Store Data in GeoJSON file 
output_directory_1 = os.path.join(project_root_dir, '1_Features/a_Spatial_Joins/2022')
output_filename_1 = "joined_ACLED_ZIP_22_PA.geojson"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

spatial_join_PA.to_file(output_path_1, driver='GeoJSON')

### Stroring and Transforming Database 

In [31]:
#Transforming Geojson file into GeoDataframe
gdf_PA = gpd.read_file(output_path_1)

#Converting Geometries to WKT
gdf_PA['geometry'] = gdf_PA['geometry'].apply(lambda x: x.wkt)

#Converting to csv + saving in Folder: b_Partial_Datasets
output_directory_2 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_2 = "joined_ACLED_ZIP_22_PA.csv"
output_path_2 = os.path.join(output_directory_2, output_filename_2)

gdf_PA.to_csv(output_path_2, index=False)

/var/folders/4s/d5gptfss61l9fqz7rk3r5m880000gn/T/ipykernel_20968/2453800663.py:6: UserWarning: Geometry column does not contain geometry.
  gdf_PA['geometry'] = gdf_PA['geometry'].apply(lambda x: x.wkt)
